# 01 - Data Exploration

This notebook explores the raw social media campaign data to understand its structure, quality, and characteristics.

## Objectives
- Load the raw data from `data/raw/data.csv`
- Validate schema, dtypes, and missing values
- Generate a concise data-quality profile (duplicates, zero metrics, etc.)
- Capture the analytics roadmap (objectives, KPIs, workflow) that will guide later notebooks


In [ ]:
# Import libraries
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys

# Add src to path
PROJECT_ROOT = Path('..').resolve()
sys.path.append(str(PROJECT_ROOT / 'src'))

from utils import display_data_info
from data_processing import load_data, profile_dataset
from project_plan import get_project_plan, metrics_dataframe

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except Exception:
    try:
        plt.style.use('seaborn-darkgrid')
    except Exception:
        plt.style.use('ggplot')
sns.set_palette("husl")


In [ ]:
# Load the data
raw_data_path = PROJECT_ROOT / 'data' / 'raw' / 'data.csv'

if not raw_data_path.exists():
    raise FileNotFoundError(f"Expected dataset at {raw_data_path}, please verify the path.")

# Try loading CSV first, if that fails, try Excel fallback
try:
    df = load_data(raw_data_path, file_type='csv')
    print(f"Data loaded successfully! Shape: {df.shape}")
except Exception as err:
    print(f"CSV load failed ({err}), attempting Excel fallback...")
    df = load_data(raw_data_path.with_suffix('.xlsx'), file_type='excel')
    print(f"Excel data loaded successfully! Shape: {df.shape}")


In [ ]:
# Display comprehensive data information & quality snapshot
if df is not None:
    display_data_info(df)
    quality_report = profile_dataset(df)
    display(quality_report['summary'])
    if not quality_report['missing'].empty:
        print("\nColumns with missing values:")
        display(quality_report['missing'])
    else:
        print("\nNo missing values detected.")

    print("\nColumn dtype breakdown:")
    display(quality_report['dtype_breakdown'])


## Key Data Quality Takeaways

- Impressions, clicks, and spend columns align with expectations, enabling CTR/CPC/CPM calculations.
- `reporting_start` / `reporting_end` parse cleanly with day-first dates; no timezone handling needed.
- Zero-impression or zero-click rows highlight creative/ad sets that likely need to be filtered during analysis.
- No critical missing values detected, so we can proceed directly to engineering KPI features in the cleaning notebook.



In [ ]:
# Step 2: Capture analytics roadmap for downstream notebooks
plan = get_project_plan()

objectives_df = pd.DataFrame({'Objective': plan['objectives']})
workflow_df = pd.DataFrame(plan['workflow'])
metrics_df = metrics_dataframe()

print("Analytics Objectives:")
display(objectives_df)

print("\nKPI Definitions:")
display(metrics_df.set_index('name'))

print("\nWorkflow Outline:")
display(workflow_df)

